In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json, csv, os, types

from pyspark.dbutils import DBUtils
dbutils = DBUtils(spark)

In [0]:
catalog_name = "medisure"
schema_config = "configurations"
schema_bronze = "bronze"
schema_silver = "silver"
schema_gold = "gold"

metadata_columns = [
    "job_id",
    "job_run_id",
    "task_run_id",
    "orchestrator_job_id",
    "orchestrator_job_run_id"
]

In [0]:
def create_table(query):
    """
    Create tables using SQL queries in config_tables.

    Args:
        query (str): SQL Query to create table.
    """
    result = spark.sql(query)
    return result

def create_widget(widget_name, default="test"):
    """
    Create widgets in our notebooks.

    Args:
        widget_name (str): Name of widget.
        default (str): Default value of widget.
    """
    return dbutils.widgets.text(widget_name, default)
    

def file_reader(file_path, file_type, header=True, sep=",", schema=None):
    """
    Read files using file path and ingest to bronze table.

    Args:
        file_path (str): File path in volume.
        file_type (str): Type of file of raw file.
        header (bool): Flag to indicate if file has header.
        sep (str): Separator for file.
        schema (str): Schema of table to be ingested to.
    """
    if file_type == "csv":
        df = spark.read.format(file_type).option("header", header).option("sep", sep).schema(schema).load(file_path)
    elif file_type == "json":
        df = spark.read.format(file_type).schema(schema).load(file_path)
    else:
        raise Exception("File type not supported")
    return df


def write_table(source_df, target_table, mode="overwrite", source_keys=None):
    """
    Write DataFrame to a target table.

    Args:
        source_df (DataFrame): Source DataFrame.
        target_table (str): Target table name (e.g., 'schema.table').
        mode (str): Write mode ('overwrite', 'append', 'ignore', 'error', 'replace').
        source_keys (list[str]): Keys to use when mode='replace' (delete+insert).

    Returns:
        DataFrame: The original DataFrame (for chaining if needed).
    """
    if mode == "overwrite":
        (source_df.write
                  .mode("overwrite")
                  .saveAsTable(target_table))

    elif mode == "append":
        (source_df.write
                  .mode("append")
                  .saveAsTable(target_table))

    elif mode == "replace":
        if not source_keys:
            raise ValueError("source_keys must be provided when mode='replace'")

        # Ensure keys are a list
        if isinstance(source_keys, str):
            source_keys = [k.strip() for k in source_keys.split(",")]

        # Build merge condition dynamically
        condition = " AND ".join([f"t.{k} = s.{k}" for k in source_keys])

        # Use Delta Lake merge for delete+insert
        from delta.tables import DeltaTable
        target = DeltaTable.forName(spark, target_table)

        (target.alias("t")
            .merge(source_df.alias("s"), condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

    else:
        raise ValueError(f"Unsupported mode: {mode}")

    return source_df
